<a href="https://colab.research.google.com/github/Phreely/Boltz-2_YAML_generator/blob/main/Boltz_2_yaml_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import requests
import yaml
import json
from string import ascii_uppercase

# User inputs
query_sequence = 'IASGGFRKYIAITGRRNVGKSSFMNALIGQEVSIVSNVAGTTTDPVFKSMELSPVGPITLIDTPGLDDVGELGIKRIKKAKKSLYRADCGILIVDDIPGNFEEQIIKLFKELEIPYFIAINKIDTIDHENIEKEYKKYNVPILKVSALKKIGFEKIGKTINSILPKDDEIPYLSDLIDGGDLVILVVPIDLGAPKGRLIMPQVHAIREGLDREALVLVVKERELRYAIENIGIKPRLVVTDSQSVMKVVSDVPEDIDLTTFSILESRYRGDLEYFVESVKAVENLKDGDTVIIMEGCTHRPLTEDIGRVKIPRWLTNHTGAALNLKVWAGVDMPELSEIEDAKLIIHCGGCVMNRNNMMRRVRMFKRLNIPMTNYGVVISYLHGVLERAIKPLMR:SNVPAELKYSKEHEWLRKEADGTYTVGITEHAQELLGDMVFVDLPEVGATVSAGDDCAVAESVKAASDIYAPVSGEIVAVNDALSDSPELVNSEPYAGGWIFKIKASDESELESLLDATAYEALLEDE'  #@param {type:"string"}
ligand_input_smiles = ''  #@param {type:"string"}
ligand_input_ccd = ''  #@param {type:"string"}
ligand_input_common_name = ''  #@param {type:"string"}
dna_input = ''  #@param {type:"string"}
jobname = 'HydF_Hmet'  #@param {type:"string"}

#@markdown ---
#@markdown ### **Advanced Settings**
#@markdown Enter a number for `max_msa`. Leave at 0 to use Boltz-2 defaults.
max_msa = 0 #@param {type:"integer"}
recycling_steps = "auto" #@param ["auto", "1", "3", "5", "10"]

#@markdown ---
#@markdown ### **Constraints Input**
#@markdown Enter constraints as a JSON array of objects (e.g., `[{"type": "contact", "atoms": ["A.1.CA", "B.2.CA"], "distance": 5.0}]`).
constraints_input = "" #@param {type:"string"}
#@markdown **Constraint Types & Rules:**
#@markdown - **`contact`**: Specifies a maximum allowed distance between two atoms. Must be in the range of 4-20 Angstroms. Example: `[{"type": "contact", "atoms": ["A.10.CA", "B.25.CB"], "distance": 8.0}]`
#@markdown - **`bond`**: Forces a covalent bond between two atoms. Example: `[{"type": "bond", "atoms": ["A.50.NZ", "LB.1.C1"]}]`
#@markdown - **`pocket`**: Defines a binding pocket for a ligand. Example: `[{"type": "pocket", "atoms": ["A.10.CA", "LB.1.C1"], "distance": 5.0}]`
#@markdown
#@markdown **Atom Formatting: `[ChainID].[ResidueNumber].[AtomName]`**
#@markdown - **Proteins**: `CA` is Alpha Carbon, `NZ` is Nitrogen in Lysine, etc. Residue number is its position in your sequence (1-indexed).
#@markdown - **Ligands**: Numbering can be tricky. It is highly recommended to run the job once without constraints, open the generated `.yaml` file to check the exact atom names and IDs assigned to your ligands, and then add your constraints!

#@markdown **Info from Github**
#@markdown `constraints` is an optional field that allows you to specify additional information about the input structure.
#@markdown    The bond constraint specifies covalent bonds between two atoms (atom1 and atom2). It is currently only supported for CCD ligands and canonical residues, CHAIN_ID refers to the id of the residue set above, RES_IDX is the index (starting from 1) of the residue (1 for ligands), and ATOM_NAME is the standardized atom name (can be verified in CIF file of that component on the RCSB website).
#@markdown
#@markdown    The pocket constraint specifies the residues associated with binding interaction, where binder refers to the chain binding to the pocket (which can be a molecule, protein, DNA or RNA) and contacts is the list of chain and residue indices (starting from 1, or atom names if the chain is a molecule) that form the binding site for the binder. max_distance specifies the maximum distance (in Angstrom, supported between 4A and 20A with 6A as default) between any atom in the binder and any atom in each of the contacts elements. If force is set to true, a potential will be used to enforce the pocket constraint.
#@markdown
#@markdown    The contact constraint specifies a contact between two residues or atoms, where token1 and token2 are the identifiers of the residues or atoms (in the format [CHAIN_ID, RES_IDX/ATOM_NAME]). max_distance specifies the maximum distance (in Angstrom, supported between 4A and 20A with 6A as default) between any pair of atoms in the two elements. If force is set to true, a potential will be used to enforce the contact constraint.


#@markdown ---
#@markdown ### **Templates**
#@markdown Specify which protein chain should be modelled using a template and provide the template PDB ID.
template_chain = "" #@param {type:"string"}
template_pdb_id = "Insert link to PDB or CIF file here" #@param {type:"string"}

#@markdown **Info from Github** `templates` is optional and allows specification of structural templates for protein chains. At minimum, provide the path to a CIF or PDB file.
#@markdown If you wish to explicitly define which of the chains in your YAML should be templated using this file, you can use the chain_id entry to specify them. If providing a PDB file, chain ids will be incrementally assigned to each subchain in a parent PDB chain resulting in template chain ids of A1, A2, B1, etc for PDB chains A and B. Make sure to look at the structure of the template PDB file to determine the corresponding value of template_id to provide. Whether a set of ids is provided or not, Boltz will find the best matching chains from the provided template. If you wish to explicitly define the mapping yourself, you may provide the corresponding template_id.
#@markdown For any template you provide, you can also specify a force flag which will use a potential to enforce that the backbone does not deviate excessively from the template during the prediction. When using force one must specify also the threshold field which controls the distance (in Angstroms) that the prediction can deviate from the template.

#@markdown ---
#@markdown ### **Affinity Prediction**
#@markdown Specify a ligand chain ID (e.g., `LB`, `CC`) to compute its binding affinity. **Note:** Must be a ligand (not a protein, DNA, or RNA) and has to be at most 128 atoms (counting heavy atoms).
compute_affinity_ligand = "" #@param {type:"string"}

# 1. Clean up and Uppercase
query_sequence = re.sub(r'\s+', '', query_sequence).upper()
dna_input = re.sub(r'\s+', '', dna_input).upper()
ligand_input_smiles = re.sub(r'\s+', '', ligand_input_smiles) # SMILES are case-sensitive, do not uppercase
ligand_input_ccd = re.sub(r'\s+', '', ligand_input_ccd).upper()
ligand_input_common_name = re.sub(r'\s+', '', ligand_input_common_name)

# 2. Setup Jobname and Directory
basejobname = re.sub(r'\W+', '', jobname)
jobname = basejobname + "_" + hashlib.sha1(query_sequence.encode()).hexdigest()[:5]
os.makedirs(jobname, exist_ok=True)

# 3. Handle Common Names via PubChem
def get_smiles(compound_name):
    try:
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"
        r = requests.get(url, timeout=5)
        return r.json()['PropertyTable']['Properties'][0]['CanonicalSMILES']
    except:
        return None

protein_sequences = query_sequence.split(':') if query_sequence else []
dna_sequences = dna_input.split(':') if dna_input else []
smiles_ligands = ligand_input_smiles.split(':') if ligand_input_smiles else []
ccd_ligands = ligand_input_ccd.split(':') if ligand_input_ccd else []

if ligand_input_common_name:
    for name in ligand_input_common_name.split(':'):
        smi = get_smiles(name)
        if smi:
            print(f"Found SMILES for {name}: {smi}")
            smiles_ligands.append(smi)

# 4. Construct YAML Dictionary
boltz_dict = {
    "name": jobname,
    "sequences": []
}

chain_gen = iter(ascii_uppercase)

# Add Proteins
for seq in protein_sequences:
    if seq:
        chain_id = next(chain_gen)
        prot_entry = {"protein": {"id": chain_id, "sequence": seq}}
        if template_chain and template_pdb_id and chain_id == template_chain.upper():
            prot_entry["protein"]["templates"] = [template_pdb_id]
        boltz_dict["sequences"].append(prot_entry)

# Add DNA
for seq in dna_sequences:
    if seq:
        boltz_dict["sequences"].append({"dna": {"id": next(chain_gen), "sequence": seq}})

# Add SMILES Ligands
for i, smi in enumerate(smiles_ligands):
    if smi:
        boltz_dict["sequences"].append({"ligand": {"id": f"L{next(chain_gen)}", "smiles": smi}})

# Add CCD Ligands
for ccd in ccd_ligands:
    if ccd:
        boltz_dict["sequences"].append({"ligand": {"id": f"C{next(chain_gen)}", "ccd": ccd}})

# 5. Advanced Parameters
sampling = {}
if recycling_steps != "auto":
    sampling["recycling_steps"] = int(recycling_steps)
if sampling:
    boltz_dict["sampling"] = sampling

if max_msa > 0:
    boltz_dict["msa"] = {"max_msa_seqs": max_msa}

# Add constraints if provided
if constraints_input.strip():
    try:
        parsed_constraints = json.loads(constraints_input)
        boltz_dict["constraints"] = parsed_constraints
    except json.JSONDecodeError:
        print("Warning: Could not parse constraints input as JSON. Skipping constraints.")

# Add affinity properties if provided
if compute_affinity_ligand.strip():
    boltz_dict["properties"] = [{"affinity": {"binder": compute_affinity_ligand.strip()}}]

# 6. Save and Download
yaml_path = os.path.join(jobname, f"{jobname}.yaml")
with open(yaml_path, 'w') as f:
    yaml.dump(boltz_dict, f, default_flow_style=False, sort_keys=False)

print(f"\nSuccess! YAML created at {yaml_path}")
files.download(yaml_path)



Success! YAML created at HydF_Hmet_b0589/HydF_Hmet_b0589.yaml


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>